# Research Library Repository Chroma Query

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.research_library_document_loader import ResearchLibraryChromaDocumentLoader
from apps.agentic.core.constants import RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = str(Path("../../.db").resolve())

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [2]:
RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('research_library',
 'research-library',
 '/Users/troy/Develop/gly.fish/yada/.db',
 ['.DS_Store', 'research_library', 'fred', 'github', 'etf'])

In [3]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [4]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['research-library']
Opened: research-library
total docs: 4875


## Verify Document Metadata

In [5]:
probe = coll.get(limit=5000)
metadata_list = probe.get("metadatas") if probe else []
metas = [m for m in metadata_list or [] if m]
len(metas)

4875

In [6]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

authors
document_date
ext
filename
h1
h2
h3
images
path
published_date
section
section_char_offset
shelf
start_index
title
topic


In [7]:
print(Counter(m.get("title") for m in metas if "title" in m and m.get("title")))
print("authors counts:", Counter(m.get("authors") for m in metas if "authors" in m and m.get("authors")))

Counter({'The Volatility Surface': 334, 'Autoregressive Processes': 278, 'Fractional Brownian Motion': 256, 'Discrete Models of Financial Markets': 250, 'Stochastic Calculus for Finance': 234, 'Monte Carlo Methods in Statistical Mechanics, Foundations and New Algorithms': 191, 'The Black-Scholes Model': 186, 'Probability Distributions': 144, 'Problems and Solutions in Mathematical Finance Stochastic Calculus Volume 1': 132, 'Options and Measure Theory Volume 4': 124, 'Options and Measure Theory Volume 2': 117, 'Hamiltonian Monte Carlo': 110, 'Bayesian Data Analysis': 102, 'Options and Measure Theory Volume 1': 102, 'Options and Measure Theory Volume 3': 94, 'Relaxation processes in a low-order three-dimensional magnetohydrodynamics model': 93, 'Marginalization and Prior Probabilities': 88, 'Analytic Mechanics': 87, 'The evolution of Alfvénic perturbations in a three-dimensional MHD model of the inner heliospheric current sheet region': 87, 'Magnetic helicity in magnetohydrodynamic turb

## Metadata Key Values

Scan the full collection to find all distinct values for each filterable metadata field.
Use these to validate filter values before passing them to the agent or similarity search.

In [8]:
# `metas` is already loaded above; reuse it here.
FILTER_KEYS = ["title", "authors", "topic", "tags"]

distinct: dict[str, set] = {k: set() for k in FILTER_KEYS}
for meta in metas:
    for key in FILTER_KEYS:
        val = meta.get(key)
        if val:
            distinct[key].add(val)

for key in FILTER_KEYS:
    values = sorted(distinct[key])
    print(f"\n{key} ({len(values)} distinct values):")
    for v in values:
        print(f"  {v!r}")


title (64 distinct values):
  "A quantum probability explanation for violations of 'rational' decision theory"
  'Analytic Mechanics'
  'Autoregressive Processes'
  'Bayesian Data Analysis'
  'Bivariate Normal Distribution'
  'Brownian Motion'
  'Continuous State Markov Chain Equilibrium'
  'Decay of Magnetic Helicity in Ideal Magnetohydrodynamics with a DC Magnetic Field'
  'Decoherent Histories Quantum Mechanics with One "Real" Fine-Grained History'
  'Discrete Cross Correlation Theorem'
  'Discrete Models of Financial Markets'
  'Discrete State Markov Chain Equilibrium'
  'Election Predictions as Martingales, An Arbitrage Approach'
  'Evaluating gambles using dynamics'
  'Fractional Brownian Motion'
  'Functions'
  'Geometry'
  'Gibbs vs Boltzmann Entropies'
  'Hamiltonian Monte Carlo'
  'How Much Data Do You Need? A Pre-asymptotic Metric for Fat-tailedness'
  'Inverse CDF Sampling'
  'Irreversible Statistical Mechanics'
  'Larger than One Probabilities in Mathematical and Practica

In [9]:
# Search within a key's distinct values to validate a specific term
search_key = "topic"
search_term = ""

matches = sorted(v for v in distinct[search_key] if search_term.lower() in v.lower())
print(f"{search_key} values containing {search_term!r}:")
for v in matches:
    print(f"  {v!r}")

topic values containing '':
  'A computational investigation of magnetic helicity of the fluctuating magnetic field in ideal and freely decaying three-dimensional (3-D) magnetohydrodynamics (MHD) in the presence of a uniform mean magnetic field is performed.'
  'A general formalism for calculation of irreversible processes, analogous to the paxtition sum algorithe of equilibrium theory, is presented. Mathematically, it is a generalization of the procedure by which Gibbs constructed his amonical and grand canonical ensembles, in which the partition function is witended to a partition functional.'
  'A recent article of Dawid, Stone, and Zidek (1973) notes two Bayesian calculation methods--the first using an improper prior, the second avoiding it--that one feels intuitively ought to lead to the same result, but in general do not. This "marginalization paradox" has been interpreted widely as revealing a fundamental inconsistency in the common Bayesian practice of using improper priors to 

## Search Examples 

In [10]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"authors": "Troy Stribling"}, {"title": "Analytic Mechanics"}]},
    },
)

hits = retriever.invoke("How are Lagrange multipliers defined?")
[print(h) for h in hits]

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


page_content='The method of Lagrange multipliers used to determithe the constraing forces acting on a system.  
Consider a system described by $n$ generalized coordinates $q_{1}, q_{2}, \ldots, q_{n}$. Furthure assume that the system has $k$ constraints acting on it. The constraints eleminate $k$ of the generalized coordinates leaving a total $n-k$ linearly independent coordinates.
The time behavior of the system is described by Hamilton's Principal  
$$
\delta I=\delta \int_{t_{s}}^{t_{f}} L d t=\int_{t_{s}}^{t_{f}} \delta L d t=0
$$  
with $k$ equations defining holonomic constraints,  
$$
f_{j}\left(q_{1}, q_{2}, \ldots, q_{n}\right)=0 \quad \forall \quad j=1,2, \ldots, k
$$  
consider the variation of the constraints  
$$
\delta f_{j}=\sum_{l=1}^{n} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$$  
For each $f_{j}$ define a scalar parameter $\lambda_{j}$ such that  
$$
\lambda_{j} \delta f_{j}=\sum_{l=1}^{n} \lambda_{j} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$

[None, None, None, None, None, None, None, None]

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

In [11]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"authors": "E. T. Jaynes"}, {"title": "The Evolution of Carnot’s Principle"}]},
    },
)
hits = retriever.invoke("What did Willard Gibbs contribute to the concept of entropy?")
[print(h) for h in hits]

page_content='Another of the curiosities of this field is that, having done so much with entropy and demonstrated such a deep understanding of the logic underlying the second law, giving thermodynamics an entirely different character, Gibbs said almost nothing about what entropy really means. He showed, far more than anyone else, how much we can accomplish by maximizing entropy. Yet we cannot learn from Gibbs: "What are we actually doing when we maximize entropy?" For this we must turn to Boltzmann.' metadata={'start_index': 7350, 'published_date': 'Jul 08, 1996', 'filename': 'carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'section': 6, 'h2': '5. THIRD METAMORPHOSIS: GIBBS', 'title': 'The Evolution of Carnot’s Principle', 'section_char_offset': 7384, 'h1': "THE EVOLUTION OF CARNOT'S PRINCIPLE*", 'shelf': 'jaynes', 'topic': 'Review of historical development of the Second Law of Thermodynamics.', 'path': 'research_library/Jaynes/carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'ext': '

[None, None, None, None, None, None, None, None]

In [20]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)
hits = retriever.invoke("What did Willard Gibbs contribute to the concept of entropy?")
[print(h) for h in hits]

page_content='Another of the curiosities of this field is that, having done so much with entropy and demonstrated such a deep understanding of the logic underlying the second law, giving thermodynamics an entirely different character, Gibbs said almost nothing about what entropy really means. He showed, far more than anyone else, how much we can accomplish by maximizing entropy. Yet we cannot learn from Gibbs: "What are we actually doing when we maximize entropy?" For this we must turn to Boltzmann.' metadata={'topic': 'Review of historical development of the Second Law of Thermodynamics.', 'section_char_offset': 7384, 'h2': '5. THIRD METAMORPHOSIS: GIBBS', 'tags': 'jaynes,thermodynamics', 'section': 6, 'ext': '.md', 'path': '/Users/troy/Develop/gly.fish/yada/research_library/Jaynes/carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'start_index': 7350, 'h1': "THE EVOLUTION OF CARNOT'S PRINCIPLE*", 'images': '', 'filename': 'carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'title': 'The 

[None, None, None, None, None, None, None, None]

## Filter Examples

In [12]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"authors": "E. T. Jaynes"}, {"title": "The Evolution of Carnot’s Principle"}]},
    },
)
hits = retriever.invoke("What did Willard Gibbs contribute to the concept of entropy?")
[print(h) for h in hits]

page_content='Another of the curiosities of this field is that, having done so much with entropy and demonstrated such a deep understanding of the logic underlying the second law, giving thermodynamics an entirely different character, Gibbs said almost nothing about what entropy really means. He showed, far more than anyone else, how much we can accomplish by maximizing entropy. Yet we cannot learn from Gibbs: "What are we actually doing when we maximize entropy?" For this we must turn to Boltzmann.' metadata={'published_date': 'Jul 08, 1996', 'h1': "THE EVOLUTION OF CARNOT'S PRINCIPLE*", 'path': 'research_library/Jaynes/carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'ext': '.md', 'start_index': 7350, 'h2': '5. THIRD METAMORPHOSIS: GIBBS', 'images': '', 'filename': 'carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'section': 6, 'shelf': 'jaynes', 'section_char_offset': 7384, 'topic': 'Review of historical development of the Second Law of Thermodynamics.', 'title': 'The Evolution of C

[None, None, None, None, None, None, None, None]